# 03 Búsqueda vectorial con TF-IDF

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

In [1]:
import re
import string
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
ruta_carpeta = Path("cache/epub")
limite_documentos = 100
puntuacion = str.maketrans("", "", string.punctuation)

def preprocesamiento_datos(texto):
    # Eliminar puntuación, convertir a minúsculas y dividir en tokens
    return texto.lower().translate(puntuacion).split()

def cargar_corpus(carpeta, limite):
    documentos = []
    for ruta in sorted(carpeta.rglob("*.txt")):
        texto = ruta.read_text(encoding="utf-8-sig", errors="replace")
        tokens = preprocesamiento_datos(texto)
        if not tokens:
            continue
        titulo = re.search(r"^Title:[ \t]*(.+)$", texto, re.MULTILINE | re.IGNORECASE)
        documentos.append({
            "id": ruta.relative_to(carpeta).as_posix(),
            "titulo": titulo[1].strip() if titulo else ruta.stem,
            "texto": texto,
            "tokens": tokens,
        })
        if len(documentos) == limite:
            break
    if not documentos:
        raise ValueError(f"No se encontraron documentos con texto en {carpeta.resolve()}")
    return documentos

In [3]:
documentos = cargar_corpus(ruta_carpeta, limite_documentos)
corpus_tokens = [doc["tokens"] for doc in documentos]
longitudes = [len(tokens) for tokens in corpus_tokens]

display(pd.DataFrame({
    "Identificador": [doc["id"] for doc in documentos],
    "Título": [doc["titulo"] for doc in documentos],
    "Términos únicos": [len(set(tokens)) for tokens in corpus_tokens],
}))
display(pd.DataFrame([{
    "Documentos": len(documentos),
    "Vocabulario": len(set().union(*corpus_tokens)),
    "Longitud mínima": min(longitudes),
    "Longitud media": np.mean(longitudes),
    "Longitud máxima": max(longitudes),
}]))

,Identificador,Título,Términos únicos
0,10/pg10.txt,The King James Version of the Bible,17083
1,100/pg100.txt,The Complete Works of William Shakespeare,33515
2,1000/pg1000.txt,La Divina Commedia di Dante: Complete,15523
3,10000/pg10000.txt,The Magna Carta,1944
4,10001/pg10001.txt,Apocolocyntosis,2176
...,...,...,...
95,10084/pg10084.txt,Kazan,5847
96,10085/pg10085.txt,Moorish Literature,9211
97,10086/pg10086.txt,The Minute Boys of the Mohawk Valley,6084
98,10087/pg10087.txt,Old Lady Number 31,5723


,Documentos,Vocabulario,Longitud mínima,Longitud media,Longitud máxima
0,100,172234,4424,73182.07,966408


### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

In [4]:
# Funcion para calcular la frecuencia de términos (TF) en un documento
def calcular_tf(tokens):
    return Counter(tokens)

# Función para calcular la frecuencia de documentos (DF) para cada término en el corpus
def calcular_df(frecuencias):
    df = Counter()
    for tf in frecuencias:
        df.update(tf.keys())
    return df

# Función para calcular el inverso de la frecuencia de documentos (IDF) para cada término
def calcular_idf(n_documentos, df):
    return {termino: np.log(n_documentos / frecuencia) + 1
            for termino, frecuencia in df.items()}
# Función para calcular el peso TF-IDF de un término en un documento
def peso_tf_idf(termino, tf, idf):
    return tf[termino] * idf[termino]

In [5]:
# Calcular las frecuencias de términos (TF) para cada documento en el corpus
frecuencias = [calcular_tf(tokens) for tokens in corpus_tokens]
df = calcular_df(frecuencias)
idf = calcular_idf(len(documentos), df)

# tres términos presentes en el primer documento, evaluados en tres documentos.
terminos_ejemplo = ["mother", "money", "father"] #ejemplo de terminos
display(pd.DataFrame([
    {"Documento": doc["id"], "Término": termino, "TF": tf[termino],
     "DF": df[termino], "IDF": idf[termino],
     "TF-IDF": peso_tf_idf(termino, tf, idf)}
    for doc, tf in zip(documentos[:3], frecuencias[:3])
    for termino in terminos_ejemplo
]))

,Documento,Término,TF,DF,IDF,TF-IDF
0,10/pg10.txt,mother,245,82,1.198451,293.620480
1,10/pg10.txt,money,143,100,1.000000,143.000000
2,10/pg10.txt,father,979,88,1.127833,1104.148871
3,100/pg100.txt,mother,351,82,1.198451,420.656279
4,100/pg100.txt,money,188,100,1.000000,188.000000
5,100/pg100.txt,father,867,88,1.127833,977.831533
6,1000/pg1000.txt,mother,0,82,1.198451,0.000000
7,1000/pg1000.txt,money,3,100,1.000000,3.000000
8,1000/pg1000.txt,father,0,88,1.127833,0.000000


### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

In [6]:
def construir_matriz(frecuencias, vocabulario, idf):
    filas, columnas, pesos = [], [], []
    for fila, tf in enumerate(frecuencias):
        for termino in tf:
            if termino in vocabulario:
                filas.append(fila)
                columnas.append(vocabulario[termino])
                pesos.append(peso_tf_idf(termino, tf, idf))
    return csr_matrix((pesos, (filas, columnas)),
                      shape=(len(frecuencias), len(vocabulario)), dtype=float)

In [7]:
vocabulario = {termino: i for i, termino in enumerate(sorted(df))}
matriz = construir_matriz(frecuencias, vocabulario, idf)
print(f"Matriz documento-término: {matriz.shape}, con {matriz.nnz:,} pesos almacenados.")

Matriz documento-término: (100, 172234), con 711,310 pesos almacenados.


### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

In [ ]:
def conservar_tokens(tokens):
    return tokens

vectorizador = TfidfVectorizer(
    analyzer=conservar_tokens, vocabulary=vocabulario,
    use_idf=True, smooth_idf=False, sublinear_tf=False, norm=None,
) #Construcción del vectorizador TF-IDF con parámetros específicos para que coincida con la implementación manual.
matriz_sklearn = vectorizador.fit_transform(corpus_tokens) 
tolerancia = 1e-10
diferencia = matriz - matriz_sklearn
error_tfidf = np.max(np.abs(diferencia.data), initial=0.0) #definir el error máximo de TF-IDF
print(f"Máxima diferencia absoluta de TF-IDF: {error_tfidf:.3g}")
assert error_tfidf <= tolerancia

Máxima diferencia absoluta de TF-IDF: 0


### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

In [9]:
def normas(matriz):
    return np.sqrt(np.asarray(matriz.multiply(matriz).sum(axis=1)).ravel())

def coseno_explicito(consultas, documentos):
    productos = (consultas @ documentos.T).toarray()
    denominador = np.outer(normas(consultas), normas(documentos))
    return np.divide(productos, denominador, out=np.zeros_like(productos),
                     where=denominador > 0)

In [10]:
consultas = {
    "whale captain ship": "Relatos sobre ballenas y navegación.",
    "king queen kingdom": "Historias sobre monarquías y sus gobernantes.",
    "faith prayer god": "Textos sobre fe y oración.",
    "war army battle": "Relatos sobre conflictos militares y batallas.",
}
display(pd.DataFrame(consultas.items(), columns=["Consulta", "Necesidad de información"]))
tokens_consultas = [preprocesamiento_datos(consulta) for consulta in consultas]
tf_consultas = [calcular_tf(tokens) for tokens in tokens_consultas]
vectores = construir_matriz(tf_consultas, vocabulario, idf)
vectores_sklearn = vectorizador.transform(tokens_consultas)
similitudes = coseno_explicito(vectores, matriz)
similitudes_sklearn = cosine_similarity(vectores_sklearn, matriz_sklearn)
error_coseno = np.max(np.abs(similitudes - similitudes_sklearn))
print(f"Máxima diferencia absoluta de coseno: {error_coseno:.3g}")
assert error_coseno <= tolerancia

def explicar_coseno(vector, documento, terminos):
    productos = vector.multiply(documento).tocoo()
    display(pd.DataFrame({
        "Término compartido": [terminos[i] for i in productos.col],
        "Producto parcial": productos.data,
    }))
    display(pd.DataFrame([{
        "Producto punto": productos.sum(),
        "Norma de la consulta": normas(vector)[0],
        "Norma del documento": normas(documento)[0],
        "Similitud": coseno_explicito(vector, documento)[0, 0],
    }]))

mejor_documento = int(np.argmax(similitudes[0]))
print(f"Consulta: {next(iter(consultas))} | Documento: {documentos[mejor_documento]['titulo']}")
explicar_coseno(vectores[0], matriz[mejor_documento], list(vocabulario))

,Consulta,Necesidad de información
0,whale captain ship,Relatos sobre ballenas y navegación.
1,king queen kingdom,Historias sobre monarquías y sus gobernantes.
2,faith prayer god,Textos sobre fe y oración.
3,war army battle,Relatos sobre conflictos militares y batallas.


Máxima diferencia absoluta de coseno: 6.38e-16
Consulta: whale captain ship | Documento: The Mystery


,Término compartido,Producto parcial
0,captain,716.820990
1,ship,238.618205


,Producto punto,Norma de la consulta,Norma del documento,Similitud
0,955.439195,3.79858,7105.652142,0.035398


### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

In [11]:
def extraer_fragmento(texto, tokens, ancho=180):
    texto = " ".join(texto.split())
    patron = "|".join(re.escape(token) for token in tokens)
    coincidencia = re.search(rf"\b(?:{patron})\b", texto, re.IGNORECASE) if patron else None
    inicio = max(0, coincidencia.start() - ancho // 3) if coincidencia else 0
    fin = inicio + ancho
    return ("…" if inicio else "") + texto[inicio:fin] + ("…" if fin < len(texto) else "")

In [12]:
def construir_ranking(puntuaciones, documentos, tokens, puntuaciones_sklearn=None):
    # En empate, se conserva el orden de carga de los documentos.
    orden = np.argsort(-puntuaciones, kind="stable")[:5]
    filas = []
    for posicion, i in enumerate(orden, start=1):
        doc = documentos[i]
        fila = {"Posición": posicion, "Identificador": doc["id"],
                "Título": doc["titulo"], "Puntuación": puntuaciones[i]}
        if puntuaciones_sklearn is not None:
            fila["Puntuación sklearn"] = puntuaciones_sklearn[i]
        fila["Fragmento"] = extraer_fragmento(doc["texto"], tokens)
        filas.append(fila)
    return pd.DataFrame(filas)


In [13]:
for i, consulta in enumerate(consultas):
    print(consulta)
    display(construir_ranking(similitudes[i], documentos, tokens_consultas[i], similitudes_sklearn[i]))

whale captain ship


,Posición,Identificador,Título,Puntuación,Puntuación sklearn,Fragmento
0,1,10008/pg10008.txt,The Mystery,0.035398,0.035398,"…IDDLE I. DESERT SEAS II. THE ""LAUGHING LASS"" ..."
1,2,10081/pg10081.txt,"The Boy Allies at Jutland; Or, The Greatest Na...",0.020676,0.020676,…eastward down the river. Men lined the rails ...
2,3,10015/pg10015.txt,"Punchinello, Volume 1, No. 19, August 6, 1870",0.020122,0.020122,"… * * * COMIC ZOOLOGY Order, Cetacea.--The Rig..."
3,4,10057/pg10057.txt,The Secret of the Tower,0.014311,0.014311,…ESSIONAL ETIQUETTE V. A FAMILIAR IMPLEMENT VI...
4,5,10070/pg10070.txt,Stories of Great Americans for Little Americans,0.012189,0.012189,…atur and the Pirates Stories about Jefferson ...


king queen kingdom


,Posición,Identificador,Título,Puntuación,Puntuación sklearn,Fragmento
0,1,100/pg100.txt,The Complete Works of William Shakespeare,0.039058,0.039058,"… THE TRAGEDY OF HAMLET, PRINCE OF DENMARK THE..."
1,2,10000/pg10000.txt,The Magna Carta,0.027799,0.027799,"…versions. The Text of Magna Carta JOHN, by th..."
2,3,10056/pg10056.txt,Chinese literature,0.022525,0.022525,"… the Analects of Confucius, the Sayings of Me..."
3,4,10/pg10.txt,The King James Version of the Bible,0.018559,0.018559,The Project Gutenberg eBook of The King James ...
4,5,10081/pg10081.txt,"The Boy Allies at Jutland; Or, The Greatest Na...",0.017179,0.017179,"… Allies with the Terror of the Seas"" 1917 CHA..."


faith prayer god


,Posición,Identificador,Título,Puntuación,Puntuación sklearn,Fragmento
0,1,10058/pg10058.txt,The Divine Office: A Study of the Roman Breviary,0.049767,0.049767,"…nts in Holy Orders, in the great and glorious..."
1,2,10/pg10.txt,The King James Version of the Bible,0.031473,0.031473,…he First Book of Moses: Called Genesis 1:1 In...
2,3,10022/pg10022.txt,White Queen of the Cannibals: the Story of Mar...,0.024981,0.024981,"…red in his drunken sleep, while the family we..."
3,4,10004/pg10004.txt,The Warriors,0.023671,0.023671,… WHAT IS WORTH WHILE? CULTURE AND REFORM THE ...
4,5,1003/pg1003.txt,"Divine Comedy, Longfellow's Translation, Paradise",0.016577,0.016577,… Mars: Martyrs and Crusaders who died fightin...


war army battle


,Posición,Identificador,Título,Puntuación,Puntuación sklearn,Fragmento
0,1,10077/pg10077.txt,"The Atlantic Monthly, Volume 10, No. 60, Octob...",0.033578,0.033578,… not believe that there were so many redcoats...
1,2,10080/pg10080.txt,Mobilizing Woman-Power,0.033223,0.033223,…ind the armies of the Allies through the year...
2,3,10078/pg10078.txt,Plain Words from America: A Letter to a German...,0.032142,0.032142,…th your opinions regarding Germany's responsi...
3,4,10062/pg10062.txt,The Iron Game: A Tale of the War,0.018449,0.018449,The Project Gutenberg eBook of The Iron Game: ...
4,5,10044/pg10044.txt,The Father of British Canada: a Chronicle of C...,0.017981,0.017981,…h. And for an English guardsman to disparage ...


### 7. Comparar con BM25

BM25 limita el efecto de repetir un término y ajusta el peso según la longitud del documento. 
Se usa un IDF propio de BM25 y se compara el conjunto de documentos de cada top-5 con el de TF-IDF. 
Las puntuaciones se interpretan dentro de cada método.

In [ ]:
def calcular_bm25(tokens, frecuencias, df, longitudes, k1=1.5, b=0.75): #Funcion obtenida de lucene.apache.org BM25Similarity.html
    n = len(frecuencias)
    ajuste = k1 * (1 - b + b * np.asarray(longitudes) / np.mean(longitudes))
    puntuaciones = np.zeros(n)
    for termino in dict.fromkeys(tokens):
        if termino not in df:
            continue
        idf_bm25 = np.log1p((n - df[termino] + 0.5) / (df[termino] + 0.5))
        tf = np.array([conteo[termino] for conteo in frecuencias])
        puntuaciones += idf_bm25 * tf * (k1 + 1) / (tf + ajuste)
    return puntuaciones


In [15]:
for i, consulta in enumerate(consultas):
    tokens = tokens_consultas[i]
    puntuaciones = calcular_bm25(tokens, frecuencias, df, longitudes)
    ranking = construir_ranking(puntuaciones, documentos, tokens)
    top_tfidf = construir_ranking(similitudes[i], documentos, tokens)
    comunes = set(ranking["Identificador"]) & set(top_tfidf["Identificador"])
    print(f"{consulta} | Coincidencias con el top-5 de TF-IDF: {len(comunes)}")
    display(ranking)

whale captain ship | Coincidencias con el top-5 de TF-IDF: 1


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,10015/pg10015.txt,"Punchinello, Volume 1, No. 19, August 6, 1870",6.651848,"… * * * COMIC ZOOLOGY Order, Cetacea.--The Rig..."
1,2,10018/pg10018.txt,"Punchinello, Volume 1, No. 05, April 30, 1870",5.416007,"…er minding whether you kill one or more, and ..."
2,3,10079/pg10079.txt,"The Atlantic Monthly, Volume 02, No. 09, July,...",5.382962,…the following charming inscription is found t...
3,4,10041/pg10041.txt,The Rivet in Grandfather's Neck: A Comedy of L...,5.043684,…y. There were no more addenda concerning the ...
4,5,10020/pg10020.txt,"The Strand Magazine: Vol. 07, Issue 37, Januar...",4.712425,… and others of Dresden. Also I note an exquis...


king queen kingdom | Coincidencias con el top-5 de TF-IDF: 4


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,100/pg100.txt,The Complete Works of William Shakespeare,4.012682,"… THE TRAGEDY OF HAMLET, PRINCE OF DENMARK THE..."
1,2,10056/pg10056.txt,Chinese literature,3.997132,"… the Analects of Confucius, the Sayings of Me..."
2,3,10000/pg10000.txt,The Magna Carta,3.974264,"…versions. The Text of Magna Carta JOHN, by th..."
3,4,10/pg10.txt,The King James Version of the Bible,3.932010,The Project Gutenberg eBook of The King James ...
4,5,10024/pg10024.txt,Beneath the Banner: Being Narratives of Noble ...,3.907280,…ached the territory of Kabbu Rega on the Vict...


faith prayer god | Coincidencias con el top-5 de TF-IDF: 3


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,10058/pg10058.txt,The Divine Office: A Study of the Roman Breviary,3.994842,"…nts in Holy Orders, in the great and glorious..."
1,2,10004/pg10004.txt,The Warriors,3.941380,… WHAT IS WORTH WHILE? CULTURE AND REFORM THE ...
2,3,10071/pg10071.txt,By-Ways of Bombay,3.902801,"… the margin of the waves. Thence, after a pau..."
3,4,10024/pg10024.txt,Beneath the Banner: Being Narratives of Noble ...,3.884521,"…Gondokoro. Still, he did not despair of accom..."
4,5,10/pg10.txt,The King James Version of the Bible,3.803764,…he First Book of Moses: Called Genesis 1:1 In...


war army battle | Coincidencias con el top-5 de TF-IDF: 3


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,10062/pg10062.txt,The Iron Game: A Tale of the War,3.258022,The Project Gutenberg eBook of The Iron Game: ...
1,2,10077/pg10077.txt,"The Atlantic Monthly, Volume 10, No. 60, Octob...",3.256737,… not believe that there were so many redcoats...
2,3,10024/pg10024.txt,Beneath the Banner: Being Narratives of Noble ...,3.199162,"…ervice in Ceylon, had been in the Crimea duri..."
3,4,10044/pg10044.txt,The Father of British Canada: a Chronicle of C...,3.179731,…h. And for an English guardsman to disparage ...
4,5,10070/pg10070.txt,Stories of Great Americans for Little Americans,3.176701,…mas Gift How Washington got out of a Trap Was...


## Conclusión
Los pesos TF-IDF calculados manualmente coincidieron con los de scikit-learn, y la diferencia en la similitud coseno fue prácticamente nula. Al comparar los cinco primeros resultados de cada consulta, TF-IDF y BM25 recuperaron algunos documentos en común unos 3 o 4 a excepción de whale captain ship que solo fue 1, aunque su orden varió. Esto muestra que la forma de ponderar los términos y ajustar por la longitud de los documentos influye en qué resultados aparecen primero.

## Acuerdo de uso de inteligencia artificial

Declaro que recurrí a la inteligencia artificial como apoyo para evaluar la estructura del notebook,aumentar la velocidad de escritura, reconocer posibles inconsistencias entre las indicaciones y el código, plantear ajustes que favorezcan la reproducibilidad y mejorar la redacción del análisis escrito. No obstante, esta herramienta no reemplaza mi criterio ni mi responsabilidad académica. Es mi deber revisar las sugerencias, ejecutar el notebook, verificar los resultados obtenidos, comprender los algoritmos implementados y referenciar las fuentes externas consultadas. Asumo la autoría y responsabilidad del trabajo final, evitando presentar como comprobados resultados que no haya validado personalmente.